In [ ]:

import os
import glob
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from scipy.stats import ttest_ind, f_oneway, mannwhitneyu, kruskal, shapiro, levene
from collections import Counter
from sklearn.preprocessing import MinMaxScaler

# Parameters
K_STATES_LIST = list(range(5, 11))
SEED = 1000
P_VALUE_THRESHOLD = 0.05

# Paths
EEG_data_dir = "../data/EEG_data/HFD_PSD_4sec_75overlap_6min"
MRI_csv_path = "../data/MRI_data/ROI_aal3_Vgm.csv_with_groups.csv"

# Output folder for NPZs
NPZ_OUT_DIR = "significant_state_npz"
os.makedirs(NPZ_OUT_DIR, exist_ok=True)

print("="*80)
print("EEG STD_DISTANCE_TO_CENTROID FEATURE ANALYSIS (with NPZ export)")
print("="*80)

# Helper functions
def file_to_sid(path_or_name: str, pad: int = 2) -> str:
    """Convert filename to standardized subject ID"""
    base = os.path.splitext(os.path.basename(path_or_name))[0]
    digits = ''.join(ch for ch in base if ch.isdigit())
    if not digits:
        if base.startswith("sub-"):
            tail = ''.join(ch for ch in base.split('-', 1)[1] if ch.isdigit())
            digits = tail or "0"
        else:
            digits = "0"
    return f"sub-{str(int(digits)).zfill(pad)}"

def _get_col(df, target_name_lower: str):
    """Find column in DataFrame ignoring case"""
    for c in df.columns:
        if c.lower() == target_name_lower:
            return c
    raise KeyError(f"Column '{target_name_lower}' not found. Available: {list(df.columns)}")

def choose_statistical_test(groups):
    """Choose appropriate statistical test"""
    if len(groups) < 2 or any(g.size == 0 for g in groups):
        return 'None', 1.0, 0.0
    try:
        normality_ok = all(shapiro(g)[1] > 0.05 for g in groups if len(g) >= 3)
        equal_var_ok = levene(*groups)[1] > 0.05
        if normality_ok and equal_var_ok:
            if len(groups) == 2:
                stat, p_val = ttest_ind(*groups)
                return 't-test', p_val, stat
            else:
                stat, p_val = f_oneway(*groups)
                return 'ANOVA', p_val, stat
        else:
            if len(groups) == 2:
                stat, p_val = mannwhitneyu(*groups)
                return 'Mann-Whitney', p_val, stat
            else:
                stat, p_val = kruskal(*groups)
                return 'Kruskal-Wallis', p_val, stat
    except Exception:
        return 'None', 1.0, 0.0

def determine_dominant_group(feature_values, labels):
    """Determine which group has highest mean"""
    label_to_group = {0: 'N', 1: 'A+P-', 2: 'A+P+'}
    group_means = {}
    for lab in np.unique(labels):
        vals = feature_values[labels == lab]
        group_means[lab] = np.mean(vals) if len(vals) else np.nan
    dominant_label = max(group_means, key=group_means.get)
    return label_to_group[dominant_label], group_means

# Data loading
print("\nStep 1: Loading subject labels and EEG data")
print("-" * 50)

# Get EEG subjects
eeg_subjects = {file_to_sid(f) for f in os.listdir(EEG_data_dir) if f.endswith(".npz")}
print(f"Found {len(eeg_subjects)} EEG subjects")

# Load MRI labels
mri_df = pd.read_csv(MRI_csv_path)
names_col = _get_col(mri_df, "names")
group_col = _get_col(mri_df, "group")

# Extract subject IDs from names
name_series = mri_df[names_col].astype(str)
subject_raw = name_series.str.extract(r'(sub-\d+)', expand=False).fillna(
    name_series.str.replace("_T1w", "", regex=False)
)
subject_num = subject_raw.str.extract(r'(\d+)', expand=False)
mri_df["subject_id"] = subject_num.apply(
    lambda d: f"sub-{str(int(d)).zfill(2)}" if pd.notna(d) and str(d).isdigit() else np.nan
)

# Create group mappings
group_to_label = {'N': 0, 'A+P-': 1, 'A+P+': 2}
mri_df["numeric_label"] = mri_df[group_col].map(group_to_label)

# Filter to subjects with both EEG and MRI data
mri_df_eeg = mri_df[mri_df["subject_id"].isin(eeg_subjects)].dropna(
    subset=["subject_id", "numeric_label"]
).copy()

risk_group_map = dict(zip(mri_df_eeg["subject_id"], mri_df_eeg["numeric_label"]))

print(f"Subjects with both EEG and MRI labels: {len(risk_group_map)}")
for grp, lab in group_to_label.items():
    cnt = sum(1 for v in risk_group_map.values() if v == lab)
    print(f"  {grp} (label={lab}): {cnt} subjects")

# Create final DataFrame
final_df = mri_df_eeg[["subject_id", group_col, "numeric_label"]].rename(
    columns={group_col: "Risk_Group", "subject_id": "participant_id", "numeric_label": "Group"}
).reset_index(drop=True)

# Load EEG features
def load_eeg_features(feature_dir: str, risk_group_map: dict):
    """Load EEG features for all subjects and windows"""
    all_features, all_labels, all_sub_ids = [], [], []
    
    print("\nLoading EEG window features...")
    for npz_path in glob.glob(os.path.join(feature_dir, "*.npz")):
        sid = file_to_sid(npz_path, pad=2)
        if sid not in risk_group_map:
            continue
            
        label = int(risk_group_map[sid])
        data = np.load(npz_path, allow_pickle=True)
        hfd = data["HFD_features"]
        n_windows = hfd.shape[0]
        subj_num = int(sid.split("-")[1])
        
        for w in range(n_windows):
            all_features.append(hfd[w].flatten())
            all_labels.append(label)
            all_sub_ids.append(subj_num)
        
        print(f"  Loaded {n_windows:4d} windows from {sid} ({['N','A+P-','A+P+'][label]})")
    
    return {
        "X": np.array(all_features),
        "y": np.array(all_labels),
        "sub_ids": np.array(all_sub_ids),
    }

eeg_data = load_eeg_features(EEG_data_dir, risk_group_map)

print(f"\nEEG Data Summary:")
print(f"  Windows: {eeg_data['X'].shape[0]}")
print(f"  Features per window: {eeg_data['X'].shape[1]}")
print(f"  Unique subjects: {len(np.unique(eeg_data['sub_ids']))}")

# store kmeans models and scaled X for retrieval later
kmeans_models = {}
X_scaled_store = {}

# Main analysis function - ONLY std_distance_to_centroid
def compute_std_distance_to_centroid_only(X_data, y_data, sub_ids_data, kmeans_model, k_states):
    """Compute ONLY std_distance_to_centroid for each subject"""
    centroids = kmeans_model.cluster_centers_
    
    # Predict all windows at once
    all_preds = kmeans_model.predict(X_data)
    
    unique_subjects = np.unique(sub_ids_data)
    subject_features = []
    
    for subject in unique_subjects:
        mask = (sub_ids_data == subject)
        if not np.any(mask):
            continue
            
        subj_X = X_data[mask]
        subj_preds = all_preds[mask]
        subj_label = y_data[mask][0]
        
        std_dist = np.full(k_states, np.nan) # Initialize with NaN
        for s in range(k_states): # For each state
            state_windows = subj_X[subj_preds == s] # Windows assigned to state s
            if state_windows.shape[0] > 0: # If there are windows for this state

                #print(f'dimesnion:{state_windows.shape}')
                distances = np.linalg.norm(state_windows - centroids[s], axis=1) # Distances to centroid
                #print(f'dimesnion:{distances.shape}')

                std_dist[s] = np.std(distances) # Std deviation of distances
                # just check variance of features in that state
                #std_dist[s] = state_windows  # np.std(np.std(state_windows, axis=0)) 
                
                # print(std_dist[s].shape)


        subject_features.append({
            'subject_id': subject,
            'label': subj_label,
            'std_distance_to_centroid': std_dist
        })
    
    return subject_features

print("\nStep 2: Feature Engineering and Statistical Testing")
print("-" * 50)

all_significant_features = []
all_features_for_wide = []  # Store ALL features for wide format

for K_STATES in K_STATES_LIST:
    print(f"\nAnalyzing K = {K_STATES}")
    print("-" * 30)
    
    # Fit KMeans - DON'T MODIFY ORIGINAL DATA
    kmeans = KMeans(n_clusters=K_STATES, random_state=SEED, n_init=10, max_iter=300)
    X_scaled = eeg_data['X'] / 2  # Create scaled copy
    kmeans.fit(X_scaled)
    
    # store fitted model & scaled X for NPZ export later
    kmeans_models[K_STATES] = kmeans
    X_scaled_store[K_STATES] = X_scaled
    
    # PCA plot
    # from sklearn.decomposition import PCA
    # import matplotlib.pyplot as plt
    # pca = PCA(n_components=2)
    # X_pca = pca.fit_transform(X_scaled)
    # plt.figure(figsize=(8,6))
    # scatter = plt.scatter(X_pca[:,0], X_pca[:,1], c=kmeans.labels_, cmap='tab10', alpha=0.6)
    # plt.title(f'KMeans Clustering (K={K_STATES}) on EEG Features')
    # plt.xlabel('PCA Component 1')
    # plt.ylabel('PCA Component 2')
    # plt.colorbar(scatter, label='Cluster Label')
    # plt.grid(True)
    # plt.show()



    # Compute std_distance_to_centroid features
    subject_features = compute_std_distance_to_centroid_only(
        X_scaled, eeg_data['y'], eeg_data['sub_ids'], kmeans, K_STATES
    )
    
    if len(subject_features) == 0:
        continue
    
    # Extract labels and feature matrix
    labels = np.array([s['label'] for s in subject_features])
    subject_ids = np.array([s['subject_id'] for s in subject_features])
    std_distance_matrix = np.vstack([s['std_distance_to_centroid'] for s in subject_features])
    
    # Test each state and store ALL features (not just significant ones)
    for state in range(K_STATES):
        vals = std_distance_matrix[:, state]
        feat_name = f"EEG_K{K_STATES}_std_distance_to_centroid_state{state}"
        
        # Store ALL features for wide format
        for subj_id, value in zip(subject_ids, vals):
            all_features_for_wide.append({
                'subject_id_num': subj_id,
                'participant_id': f"sub-{int(subj_id):02d}",
                'feature_name': feat_name,
                'value': value,
                'K': K_STATES,
                'state': state
            })
        
        # Handle NaN values for statistical testing
        valid_mask = ~np.isnan(vals)
        if not np.any(valid_mask):
            continue
            
        valid_vals = vals[valid_mask]
        valid_labels = labels[valid_mask]
        valid_subject_ids = subject_ids[valid_mask]
        
        if len(np.unique(valid_labels)) < 2:
            continue
        
        # Statistical test
        groups = [valid_vals[valid_labels == lab] for lab in np.unique(valid_labels)]
        test_name, p_value, _ = choose_statistical_test(groups)
        
        if p_value < P_VALUE_THRESHOLD:
            dom_group, group_means = determine_dominant_group(valid_vals, valid_labels)
            print(f" ✓ State {state} std_distance: p={p_value:.6f} ({dom_group} higher)")
            
            all_significant_features.append({
                'K': K_STATES,
                'feature_type': 'std_distance_to_centroid',
                'state': state,
                'p_value': p_value,
                'test': test_name,
                'dominant_group': dom_group,
                'group_means': group_means,
                'feature_values': vals,  # Include ALL values (with NaN)
                'subject_ids': subject_ids  # Include ALL subject IDs
            })


# correct p values here in all_significant_features
from statsmodels.stats.multitest import multipletests
if all_significant_features:
    p_values = [feat['p_value'] for feat in all_significant_features]
    reject, pvals_corrected, _, _ = multipletests(p_values, alpha=P_VALUE_THRESHOLD, method='fdr_bh')
    
    for i, feat in enumerate(all_significant_features):
        feat['p_value_corrected'] = pvals_corrected[i]
        feat['significant_after_correction'] = reject[i]
    
    print("\nCorrected p-values using FDR-BH:")
    for feat in all_significant_features:
        sig_marker = "✓" if feat['significant_after_correction'] else "✗"
        print(f" {sig_marker} K={feat['K']} State {feat['state']} std_distance: corrected p={feat['p_value_corrected']:.6f}")
else:
    print("\nNo significant features found to correct.")

# create a dataframe for only significant results after correction
significant_results_corrected = [feat for feat in all_significant_features if feat['significant_after_correction']]




In [ ]:
import numpy as np
import os 
BASE_DIR = "../data"
CONN_DIR = os.path.join(BASE_DIR, "fMRI_data/subject_windows_confounds_old_60sec")

list_of_files = os.listdir(CONN_DIR)
print(list_of_files)

all_data = []
all_labels = []
for file in list_of_files:
    sub_data = []
    if file.endswith(".npz"):
        data = np.load(os.path.join(CONN_DIR, file))
        
        AP_temp = data["AP"]
        PA_temp = data["PA"]
        label = data["label"]
        
        
        sub_data = np.concatenate((AP_temp, PA_temp), axis=0)
        
        all_data.append(sub_data)
        all_labels.append(label)

all_data = np.array(all_data)
all_labels = np.array(all_labels)

print(all_data.shape)
print(all_labels.shape)
        

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans

# Parameters
K_RANGE = range(2, 16)  # Extended range for better elbow visualization
SEED = 1

print("="*80)
print("ELBOW CURVE ANALYSIS: EEG vs fMRI")
print("="*80)

# ============= EEG INERTIA CALCULATION =============
print("\nCalculating EEG inertia...")
eeg_inertia = []
X_eeg_scaled = eeg_data['X'] / 2  # Same scaling as your analysis

for k in K_RANGE:
    kmeans_eeg = KMeans(n_clusters=k, random_state=SEED, n_init="auto", max_iter=10000)
    kmeans_eeg.fit(X_eeg_scaled)
    eeg_inertia.append(kmeans_eeg.inertia_)
    print(f"  K={k:2d}, Inertia={kmeans_eeg.inertia_:.2f}")

# Normalize EEG inertia to 0-1 scale
eeg_inertia_normalized = np.array(eeg_inertia)
eeg_inertia_normalized = (eeg_inertia_normalized - eeg_inertia_normalized.min()) / \
                          (eeg_inertia_normalized.max() - eeg_inertia_normalized.min())

# ============= fMRI INERTIA CALCULATION =============
print("\nCalculating fMRI inertia...")
fmri_inertia = []
n_subjects, n_timepoints, n_features = all_data.shape
X_fmri_scaled = all_data.reshape(-1, n_features)  # Same as your fMRI code

for k in K_RANGE:
    kmeans_fmri = KMeans(n_clusters=k, random_state=SEED, n_init="auto", max_iter=10000)
    kmeans_fmri.fit(X_fmri_scaled)
    fmri_inertia.append(kmeans_fmri.inertia_)
    print(f"  K={k:2d}, Inertia={kmeans_fmri.inertia_:.2f}")

# Normalize fMRI inertia to 0-1 scale
fmri_inertia_normalized = np.array(fmri_inertia)
fmri_inertia_normalized = (fmri_inertia_normalized - fmri_inertia_normalized.min()) / \
                           (fmri_inertia_normalized.max() - fmri_inertia_normalized.min())

# ============= PLOT ELBOW CURVES =============
print("\nCreating elbow curve plot...")

plt.figure(figsize=(10, 6))
plt.plot(list(K_RANGE), eeg_inertia_normalized, 'o-', color='#4A90E2', 
         linewidth=2, markersize=8, label='EEG Inertia (Scaled)', alpha=0.8)
plt.plot(list(K_RANGE), fmri_inertia_normalized, 's-', color='#E24A4A', 
         linewidth=2, markersize=8, label='fMRI Inertia (Scaled)', alpha=0.8)

# Optional: Mark the "knee" points (K=6 for both based on your analysis range)
knee_k_eeg = 6
knee_k_fmri = 6

# Find indices for knee points
knee_idx_eeg = list(K_RANGE).index(knee_k_eeg)
knee_idx_fmri = list(K_RANGE).index(knee_k_fmri)

plt.plot(knee_k_eeg, eeg_inertia_normalized[knee_idx_eeg], 'X', 
         markersize=15, color='#4A90E2', markeredgewidth=3, 
         markeredgecolor='darkblue', label='EEG Knee (K=6)')
plt.plot(knee_k_fmri, fmri_inertia_normalized[knee_idx_fmri], 'X', 
         markersize=15, color='#E24A4A', markeredgewidth=3, 
         markeredgecolor='darkred', label='fMRI Knee (K=6)')

# Styling
plt.xlabel('Number of Clusters (K)', fontsize=14, fontweight='bold')
plt.ylabel('Normalized Inertia (Within-Cluster Sum of Squares)', fontsize=14, fontweight='bold')
plt.title('EEG and fMRI Elbow Curve Plot', fontsize=16, fontweight='bold')
plt.legend(fontsize=11, loc='upper right', framealpha=0.9)
plt.grid(True, alpha=0.3, linestyle='--')
plt.xticks(list(K_RANGE))
plt.tight_layout()

# Save figure
output_plot = "elbow_curve_eeg_fmri_comparison.png"
plt.savefig(output_plot, dpi=300, bbox_inches='tight', facecolor='white')
print(f"\n✅ Saved elbow curve plot to: {output_plot}")
plt.show()

# ============= PRINT STATISTICS =============
print("\n" + "="*80)
print("ELBOW ANALYSIS SUMMARY")
print("="*80)

print("\nEEG Normalized Inertia:")
for k, inertia in zip(K_RANGE, eeg_inertia_normalized):
    marker = " 👈" if k == knee_k_eeg else ""
    print(f"  K={k:2d}: {inertia:.4f}{marker}")

print("\nfMRI Normalized Inertia:")
for k, inertia in zip(K_RANGE, fmri_inertia_normalized):
    marker = " 👈" if k == knee_k_fmri else ""
    print(f"  K={k:2d}: {inertia:.4f}{marker}")

# Calculate percentage drops
eeg_drop_at_6 = (eeg_inertia_normalized[list(K_RANGE).index(5)] - 
                 eeg_inertia_normalized[knee_idx_eeg]) / eeg_inertia_normalized[list(K_RANGE).index(5)] * 100
fmri_drop_at_6 = (fmri_inertia_normalized[list(K_RANGE).index(5)] - 
                  fmri_inertia_normalized[knee_idx_fmri]) / fmri_inertia_normalized[list(K_RANGE).index(5)] * 100

print(f"\nInertia reduction from K=5 to K=6:")
print(f"  EEG:  {eeg_drop_at_6:.2f}%")
print(f"  fMRI: {fmri_drop_at_6:.2f}%")

print(f"\n✅ Analysis complete! Both modalities show elbow around K=6-7")

In [ ]:
from kneed import KneeLocator
import numpy as np
fmri_inertia = np.array(fmri_inertia)

# Determine knees using kneed
eeg_knee = KneeLocator(list(K_RANGE), eeg_inertia_normalized, curve='convex', direction='decreasing').knee
fmri_knee = KneeLocator(list(K_RANGE), fmri_inertia, curve='convex', direction='decreasing').knee
print(f"\nKnee points detected by kneed:")
print(f"  EEG Knee: K={eeg_knee}")
print(f"  fMRI Knee: K={fmri_knee}")

In [ ]:
from kneed import KneeLocator
import numpy as np

eeg_inertia = np.asarray(eeg_inertia).ravel()
fmri_inertia = np.asarray(fmri_inertia).ravel()

k_values_eeg = list(range(2, 2 + len(eeg_inertia)))
k_values_fmri = list(range(2, 2 + len(fmri_inertia)))

eeg_inertia_norm = (eeg_inertia - eeg_inertia.min()) / (eeg_inertia.max() - eeg_inertia.min())
fmri_inertia_norm = (fmri_inertia - fmri_inertia.min()) / (fmri_inertia.max() - fmri_inertia.min())

eeg_knee = KneeLocator(
    k_values_eeg,
    eeg_inertia_norm,
    curve='convex',
    direction='decreasing'
).knee

fmri_knee = KneeLocator(
    k_values_fmri,
    fmri_inertia_norm,
    curve='convex',
    direction='decreasing'
).knee

print("\nKnee points detected by kneed:")
print(f"  EEG Knee: K={eeg_knee}")
print(f"  fMRI Knee: K={fmri_knee}")


In [ ]:
from kneed import KneeLocator
import numpy as np

# Convert to 1D numpy arrays
eeg_inertia = np.asarray(eeg_inertia).ravel()
fmri_inertia = np.asarray(fmri_inertia).ravel()

# Build K values from the actual array lengths
# Assumes your clustering started at K=2
k_values_eeg = list(range(2, 2 + len(eeg_inertia)))
k_values_fmri = list(range(2, 2 + len(fmri_inertia)))

# Use the same approach for both modalities
# Option 1: raw inertia for both
eeg_knee = KneeLocator(
    k_values_eeg,
    eeg_inertia,
    curve='convex',
    direction='decreasing'
).knee

fmri_knee = KneeLocator(
    k_values_fmri,
    fmri_inertia,
    curve='convex',
    direction='decreasing'
).knee

print("\nKnee points detected by kneed:")
print(f"  EEG Knee: K={eeg_knee}")
print(f"  fMRI Knee: K={fmri_knee}")


In [ ]:
# ============= DETECT KNEE POINTS =============
eeg_knee = KneeLocator(list(K_RANGE), eeg_inertia_normalized, curve='convex', direction='decreasing').knee
fmri_knee = KneeLocator(list(K_RANGE), fmri_inertia_normalized, curve='convex', direction='decreasing').knee

print(f"\nKnee points detected by kneed:")
print(f"  EEG Knee: K={eeg_knee}")
print(f"  fMRI Knee: K={fmri_knee}")

# Define next 6 K values after knee
eeg_highlight_range = range(eeg_knee, eeg_knee + 6)
fmri_highlight_range = range(fmri_knee, fmri_knee + 6)

print(f"\nHighlighting ranges:")
print(f"  EEG: K={eeg_knee} to K={eeg_knee+5}")
print(f"  fMRI: K={fmri_knee} to K={fmri_knee+5}")

# ============= PLOT ELBOW CURVES =============
print("\nCreating elbow curve plot...")

fig, ax = plt.subplots(figsize=(10, 6))

# Softer color palette
eeg_line_color = '#7EB5D6'      # Soft blue
fmri_line_color = '#E88D8D'     # Soft coral/pink
eeg_shade_color = "#89C5F0"     # Very light blue
fmri_shade_color = "#F19B9B"    # Very light coral
eeg_marker_color = '#2E7DAF'    # Deeper blue for contrast
fmri_marker_color = '#C75050'   # Deeper coral for contrast

# Plot main curves
ax.plot(list(K_RANGE), eeg_inertia_normalized, 'o-', color=eeg_line_color, 
        linewidth=2, markersize=8, label='EEG Inertia (Scaled)', alpha=0.85)
ax.plot(list(K_RANGE), fmri_inertia_normalized, 's-', color=fmri_line_color, 
        linewidth=2, markersize=8, label='fMRI Inertia (Scaled)', alpha=0.85)

# Highlight the next 6 K values after knee with shaded region
ax.axvspan(eeg_knee, eeg_knee + 6, alpha=0.2, color=eeg_shade_color, 
           label=f'EEG Range (K={eeg_knee}-{eeg_knee+5})', zorder=0)
ax.axvspan(fmri_knee, fmri_knee + 6, alpha=0.2, color=fmri_shade_color, 
           label=f'fMRI Range (K={fmri_knee}-{fmri_knee+5})', zorder=0)

# Mark knee points with O and X
knee_idx_eeg = list(K_RANGE).index(eeg_knee)
knee_idx_fmri = list(K_RANGE).index(fmri_knee)

ax.plot(eeg_knee, eeg_inertia_normalized[knee_idx_eeg], "x",
        markersize=16, color='white', markeredgewidth=2.0, 
        markerfacecolor='none', markeredgecolor=eeg_marker_color, 
        label=f'EEG Knee (K={eeg_knee})', zorder=5)
ax.plot(fmri_knee, fmri_inertia_normalized[knee_idx_fmri], "x",
        markersize=16, color=fmri_marker_color, markeredgewidth=2.0, 
        label=f'fMRI Knee (K={fmri_knee})', zorder=5)

# Styling
ax.set_xlabel('Number of Clusters (K)', fontsize=14)
ax.set_ylabel('Normalized Inertia (Within-Cluster Sum of Squares)', fontsize=14)
# ax.set_title('EEG and fMRI Elbow Curve Plot', fontsize=16)
ax.legend(fontsize=10, loc='upper right', framealpha=0.95)
ax.grid(True, alpha=0.3, linestyle='--', color='gray')
ax.set_xticks(list(K_RANGE))
plt.tight_layout()

# Save figure
output_plot = "elbow_curve_eeg_fmri_comparison.png"
plt.savefig(output_plot, dpi=300, bbox_inches='tight', facecolor='white')
print(f"\n✅ Saved elbow curve plot to: {output_plot}")
plt.show()

# ============= PRINT STATISTICS =============
print("\n" + "="*80)
print("ELBOW ANALYSIS SUMMARY")
print("="*80)

print("\nEEG Normalized Inertia:")
for k, inertia in zip(K_RANGE, eeg_inertia_normalized):
    marker = " 👈 KNEE" if k == eeg_knee else (" ✓ Selected" if k in eeg_highlight_range else "")
    print(f"  K={k:2d}: {inertia:.4f}{marker}")

print("\nfMRI Normalized Inertia:")
for k, inertia in zip(K_RANGE, fmri_inertia_normalized):
    marker = " 👈 KNEE" if k == fmri_knee else (" ✓ Selected" if k in fmri_highlight_range else "")
    print(f"  K={k:2d}: {inertia:.4f}{marker}")

print(f"\n✅ Analysis complete!")
print(f"   EEG uses K={eeg_knee} to K={eeg_knee+5} (shaded in soft blue)")
print(f"   fMRI uses K={fmri_knee} to K={fmri_knee+5} (shaded in soft coral)")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans
from kneed import KneeLocator


import numpy as np
import os 
BASE_DIR = "../data"
CONN_DIR = os.path.join(BASE_DIR, "fMRI_data/subject_windows_confounds_old_60sec")

list_of_files = os.listdir(CONN_DIR)
print(list_of_files)

all_data = []
all_labels = []
for file in list_of_files:
    sub_data = []
    if file.endswith(".npz"):
        data = np.load(os.path.join(CONN_DIR, file))
        
        AP_temp = data["AP"]
        PA_temp = data["PA"]
        label = data["label"]
        
        
        sub_data = np.concatenate((AP_temp, PA_temp), axis=0)
        
        all_data.append(sub_data)
        all_labels.append(label)

all_data = np.array(all_data)
all_labels = np.array(all_labels)

print(all_data.shape)
print(all_labels.shape)
        

# ================= PARAMETERS =================
K_RANGE = range(2, 15)   # K = 2..15
SEED = 1
N_INIT = "auto"
MAX_ITER = 10000

print("=" * 80)
print("ELBOW CURVE ANALYSIS: EEG vs fMRI")
print("=" * 80)

# ================= PREPARE DATA =================
X_eeg = eeg_data["X"] / 2
n_subjects, n_timepoints, n_features = all_data.shape
X_fmri = all_data.reshape(-1, n_features)

# ================= COMPUTE INERTIA =================
print("\nCalculating EEG inertia...")
eeg_inertia = []
for k in K_RANGE:
    km = KMeans(
        n_clusters=k,
        random_state=SEED,
        n_init=N_INIT,
        max_iter=MAX_ITER
    )
    km.fit(X_eeg)
    eeg_inertia.append(km.inertia_)
    print(f"  EEG  K={k:2d}, Inertia={km.inertia_:.2f}")

print("\nCalculating fMRI inertia...")
fmri_inertia = []
for k in K_RANGE:
    km = KMeans(
        n_clusters=k,
        random_state=SEED,
        n_init=N_INIT,
        max_iter=MAX_ITER
    )
    km.fit(X_fmri)
    fmri_inertia.append(km.inertia_)
    print(f"  fMRI K={k:2d}, Inertia={km.inertia_:.2f}")

eeg_inertia = np.asarray(eeg_inertia, dtype=float)
fmri_inertia = np.asarray(fmri_inertia, dtype=float)

# ================= NORMALIZE =================
def minmax_scale(x):
    x_min, x_max = x.min(), x.max()
    if np.isclose(x_max, x_min):
        return np.zeros_like(x)
    return (x - x_min) / (x_max - x_min)

eeg_inertia_normalized = minmax_scale(eeg_inertia)
fmri_inertia_normalized = minmax_scale(fmri_inertia)

# ================= DETECT KNEE POINTS =================
k_values = list(K_RANGE)

eeg_knee = KneeLocator(
    k_values,
    eeg_inertia_normalized,
    curve="convex",
    direction="decreasing"
).knee

fmri_knee = KneeLocator(
    k_values,
    fmri_inertia_normalized,
    curve="convex",
    direction="decreasing"
).knee

print(f"\nKnee points detected by kneed:")
print(f"  EEG Knee: K={eeg_knee}")
print(f"  fMRI Knee: K={fmri_knee}")

# Optional fallback if kneed returns None
if eeg_knee is None:
    eeg_knee = k_values[0]
if fmri_knee is None:
    fmri_knee = k_values[0]

# Highlight knee through next 5 K values, clipped to available range
max_k = max(k_values)
eeg_range_end = min(eeg_knee + 5, max_k)
fmri_range_end = min(fmri_knee + 5, max_k)

eeg_highlight_range = range(eeg_knee, eeg_range_end + 1)
fmri_highlight_range = range(fmri_knee, fmri_range_end + 1)

print(f"\nHighlighting ranges:")
print(f"  EEG: K={eeg_knee} to K={eeg_range_end}")
print(f"  fMRI: K={fmri_knee} to K={fmri_range_end}")

# ================= PLOT =================
print("\nCreating elbow curve plot...")

fig, ax = plt.subplots(figsize=(10, 6))

eeg_line_color = "#7EB5D6"
fmri_line_color = "#E88D8D"
eeg_shade_color = "#89C5F0"
fmri_shade_color = "#F19B9B"
eeg_marker_color = "#2E7DAF"
fmri_marker_color = "#C75050"

ax.plot(
    k_values, eeg_inertia_normalized, "o-",
    color=eeg_line_color, linewidth=2, markersize=8,
    label="EEG Inertia (Normalized)", alpha=0.85
)
ax.plot(
    k_values, fmri_inertia_normalized, "s-",
    color=fmri_line_color, linewidth=2, markersize=8,
    label="fMRI Inertia (Normalized)", alpha=0.85
)

# Shade integer K regions cleanly
ax.axvspan(
    eeg_knee - 0.5, eeg_range_end + 0.5,
    alpha=0.2, color=eeg_shade_color,
    label=f"EEG Range (K={eeg_knee}-{eeg_range_end})", zorder=0
)
ax.axvspan(
    fmri_knee - 0.5, fmri_range_end + 0.5,
    alpha=0.2, color=fmri_shade_color,
    label=f"fMRI Range (K={fmri_knee}-{fmri_range_end})", zorder=0
)

eeg_knee_idx = k_values.index(eeg_knee)
fmri_knee_idx = k_values.index(fmri_knee)

ax.plot(
    eeg_knee, eeg_inertia_normalized[eeg_knee_idx], "x",
    markersize=14, color=eeg_marker_color, markeredgewidth=2.5,
    label=f"EEG Knee (K={eeg_knee})", zorder=5
)
ax.plot(
    fmri_knee, fmri_inertia_normalized[fmri_knee_idx], "x",
    markersize=14, color=fmri_marker_color, markeredgewidth=2.5,
    label=f"fMRI Knee (K={fmri_knee})", zorder=5
)

ax.set_xlabel("Number of Clusters (K)", fontsize=14)
ax.set_ylabel("Normalized Inertia", fontsize=14)
ax.set_xticks(k_values)
ax.grid(True, alpha=0.3, linestyle="--", color="gray")
ax.legend(fontsize=10, loc="upper right", framealpha=0.95)

plt.tight_layout()
output_plot = "elbow_curve_eeg_fmri_comparison.png"
plt.savefig(output_plot, dpi=300, bbox_inches="tight", facecolor="white")
print(f"\nSaved elbow curve plot to: {output_plot}")
plt.show()

# ================= SUMMARY =================
print("\n" + "=" * 80)
print("ELBOW ANALYSIS SUMMARY")
print("=" * 80)

print("\nEEG Normalized Inertia:")
for k, inertia in zip(k_values, eeg_inertia_normalized):
    marker = " <- KNEE" if k == eeg_knee else (" <- Selected" if k in eeg_highlight_range else "")
    print(f"  K={k:2d}: {inertia:.4f}{marker}")

print("\nfMRI Normalized Inertia:")
for k, inertia in zip(k_values, fmri_inertia_normalized):
    marker = " <- KNEE" if k == fmri_knee else (" <- Selected" if k in fmri_highlight_range else "")
    print(f"  K={k:2d}: {inertia:.4f}{marker}")

print("\nAnalysis complete.")
print(f"EEG uses K={eeg_knee} to K={eeg_range_end}")
print(f"fMRI uses K={fmri_knee} to K={fmri_range_end}")


In [ ]:
from kneed import KneeLocator
import numpy as np

k_values = list(range(2, 15))  # same as before
fmri_inertia_sub = np.asarray(fmri_inertia[:len(k_values)])

fmri_knee = KneeLocator(
    k_values,
    fmri_inertia_sub,
    curve="convex",
    direction="decreasing"
).knee

print(f"fMRI Knee: K={fmri_knee}")

# for eeg
eeg_inertia_sub = np.asarray(eeg_inertia[:len(k_values)])
eeg_knee = KneeLocator(
    k_values,
    eeg_inertia_sub,
    curve="convex",
    direction="decreasing"
).knee  
print(f"EEG Knee: K={eeg_knee}")
